In [1]:
import numpy as np

tell whether a given mushroom is edible or poisonous based on it's physical attributes
-  You have 10 examples of mushrooms. For each example, you have
    - Three features
        - Cap Color (`Brown` or `Red`),
        - Stalk Shape (`Tapering (as in \/)` or `Enlarging (as in /\)`), and
        - Solitary (`Yes` or `No`)
    - Label
        - Edible (`1` indicating yes or `0` indicating poisonous)
Therefore,
- `X_train` contains three features for each example
    - Brown Color (A value of `1` indicates "Brown" cap color and `0` indicates "Red" cap color)
    - Tapering Shape (A value of `1` indicates "Tapering Stalk Shape" and `0` indicates "Enlarging" stalk shape)
    - Solitary  (A value of `1` indicates "Yes" and `0` indicates "No")

- `y_train` is whether the mushroom is edible
    - `y = 1` indicates edible
    - `y = 0` indicates poisonous


In [2]:
## data set
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

### calculate entropy
$$H(p_1) = -p_1 \text{log}_2(p_1) - (1- p_1) \text{log}_2(1- p_1)$$


In [3]:
def entropy(y):
    entropy = 0
    if len(y) != 0:
        p1 = len(y[y==1]) / len(y)
        if p1 !=0 and p1!=1:
            entropy = -p1*np.log2(p1) - (1-p1)*np.log2(1-p1)
        else :
            entropy = 0
    return entropy

### Split dataset

In [4]:
def split_data(X,node_indeces,feature):
    left_side =[]
    right_side =[]

    for i in  node_indeces:
        if X[i][feature] ==1 :
            left_side.append(i)
        else:
            right_side.append(i)
    return left_side,right_side

### Calculate information gain
$$\text{Information Gain} = H(p_1^\text{node})- (w^{\text{left}}H(p_1^\text{left}) + w^{\text{right}}H(p_1^\text{right}))$$


In [5]:
def info_gain(X,Y,node_indi,feature):

    left_indi,right_indi = split_data(X,node_indi,feature)

    if len(left_indi) == 0 or len(right_indi) == 0:         #edge case
        return 0

    y_node = Y[node_indi]
    y_left = Y[left_indi]
    y_right = Y[right_indi]
    information_gain = 0
    node_entropy = entropy(y_node)
    left_entropy = entropy(y_left)
    right_entropy = entropy(y_right)

    w_left = len(left_indi)/len(node_indi)
    w_right = len(right_indi)/len(node_indi)

    weighted_entropy = w_left*left_entropy+w_right*right_entropy

    information_gain = node_entropy - weighted_entropy

    return information_gain


In [6]:
root_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
info_gain0 = info_gain(X_train, y_train, root_indices, feature=0)

print("Information Gain from splitting the root on brown cap: ", info_gain0)

info_gain1 = info_gain(X_train, y_train, root_indices, feature=1)
print("Information Gain from splitting the root on tapering stalk shape: ", info_gain1)

info_gain2 = info_gain(X_train, y_train, root_indices, feature=2)
print("Information Gain from splitting the root on solitary: ", info_gain2)


Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377


In [7]:
def get_best_split(X,Y,node_indi,used_features):
    num_festures = X.shape[1]
    best_feature = -1
    max_info_gain =0
    for feature in range(num_festures):
        if feature in used_features:
            continue
        information_gain = info_gain(X,Y,node_indi, feature)
        if information_gain > max_info_gain:
            max_info_gain = information_gain
            best_feature = feature

    return best_feature

In [8]:
def build_tree(X,Y,node_indices,branch_name,current_dept,max_depth,used_features):

    if len(set(Y[node_indices]))==1:
        formatting = "-" * current_dept
        print(formatting, f"{branch_name} leaf (pure), class =", Y[node_indices][0])
        return

    if current_dept == max_depth:
        formatting = " "*current_dept + "-"*current_dept
        print(formatting, f"{branch_name} leaf node with indices" , node_indices)

        return


    best_feature = get_best_split(X,Y,node_indices,used_features)
    used_features.add(best_feature)
    formatting = "-"*current_dept
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_dept, branch_name, best_feature))
    left_side , right_side = split_data(X,node_indices,best_feature)

    if(len(left_side)==0 or len(right_side)==0):
        formatting = "-" * current_dept
        print(formatting, f"{branch_name} leaf (no valid split)")
        return
    build_tree(X,Y,left_side, "left", current_dept+1, max_depth,used_features.copy())
    build_tree(X,Y,right_side, "right", current_dept+1, max_depth,used_features.copy())


In [9]:

build_tree(X_train, y_train, root_indices, "Root", max_depth=2, current_dept=0,used_features=set())

 Depth 0, Root: Split on feature: 2
- Depth 1, left: Split on feature: 0
-- left leaf (pure), class = 1
-- right leaf (pure), class = 0
- Depth 1, right: Split on feature: 1
-- left leaf (pure), class = 1
-- right leaf (pure), class = 0
